# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study
프로젝트 루트: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study
원본 데이터 폴더: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\practice\chapter05\data\raw
전처리 데이터 폴더: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\data\processed
보고서 폴더: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


       dataset  rows  columns
0    customers   152        6
1     products   102        4
2       orders   302        5
3  order_items   765        5

In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


             missing_count  missing_ratio
age                      4           2.63
city                     2           1.32
name                     0           0.00
customer_id              0           0.00
gender                   0           0.00
signup_date              0           0.00

## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


             missing_count  missing_ratio
customer_id              0            0.0
name                     0            0.0
gender                   0            0.0
age                      0            0.0
city                     0            0.0
signup_date              0            0.0

## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


  order_date order_month order_dayofweek
0 2026-07-12     2026-07          Sunday
1 2025-10-02     2025-10        Thursday
2 2026-03-13     2026-03          Friday
3 2025-10-12     2025-10          Sunday
4 2026-04-17     2026-04          Friday

In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


   customer_id signup_date
0            1  2025-10-05
1            2  2023-09-20
2            3  2025-01-22
3            4  2025-03-13
4            5  2024-02-10

## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [18]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06

         quantity     unit_price
count  758.000000     761.000000
mean     3.168865  103494.086728
std      4.159055   56969.823769
min     -1.000000       0.000000
25%      2.000000   55000.000000
50%      3.000000   96000.000000
75%      4.000000  154000.000000
max    100.000000  197000.000000

In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [20]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


   quantity  unit_price  line_total
0       2.0    172000.0    344000.0
1       4.0     79000.0    316000.0
2       3.0     87000.0    261000.0
3       1.0    187000.0    187000.0
4       5.0    125000.0    625000.0

In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [23]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


                                                  check  invalid_count
0    orders.customer_id exists in customers.customer_id              2
1        order_items.order_id exists in orders.order_id              2
2  order_items.product_id exists in products.product_id             26

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [24]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


       dataset  rows_raw  columns_raw  rows_processed  columns_processed
0    customers       152            6             150                  6
1     products       102            4              96                  4
2       orders       302            5             300                  7
3  order_items       765            5             752                  6

## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [25]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/data/processed/customers_clean.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/data/processed/orders_clean.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/data/processed/order_items_clean.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [26]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [27]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [28]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [29]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


       dataset  rows_raw  columns_raw  rows_processed  columns_processed
0    customers       152            6             150                  6
1  order_items       765            5             752                  6
2       orders       302            5             300                  7
3     products       102            4              96                  4

In [30]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


       dataset  row_duplicate_count     key_column  key_duplicate_count
0    customers                    0    customer_id                    0
1     products                    0     product_id                    0
2       orders                    0       order_id                    0
3  order_items                    0  order_item_id                    0

In [31]:
module_relationship_checks


                                                  check  invalid_count
0    orders.customer_id exists in customers.customer_id              2
1        order_items.order_id exists in orders.order_id              2
2  order_items.product_id exists in products.product_id             26

## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [32]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합칩니다.
missing_tables = []
for name, df in clean_data.items():
    table = missing_summary(df).reset_index(names='column')
    table.insert(0, 'dataset', name)
    missing_tables.append(table)
missing_rates = pd.concat(missing_tables, ignore_index=True)
missing_rates


        dataset          column  missing_count  missing_ratio
0     customers     customer_id              0            0.0
1     customers            name              0            0.0
2     customers          gender              0            0.0
3     customers             age              0            0.0
4     customers            city              0            0.0
5     customers     signup_date              0            0.0
6      products      product_id              0            0.0
7      products    product_name              0            0.0
8      products        category              0            0.0
9      products           price              0            0.0
10       orders        order_id              0            0.0
11       orders     customer_id              0            0.0
12       orders      order_date              0            0.0
13       orders  payment_method              0            0.0
14       orders    order_status              0            0.0
15  orde

In [33]:
# 과제 2. 18세 미만 또는 100세 초과 고객을 확인합니다.
out_of_range_customers = customers_clean.loc[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100),
    ['customer_id', 'age'],
]
print('후보 건수:', len(out_of_range_customers))
out_of_range_customers


후보 건수: 1


    customer_id    age
32           33  150.0

In [34]:
# 과제 3. 유효한 주문 날짜가 있는 주문을 월별로 집계합니다.
monthly_order_counts = (
    orders_clean.dropna(subset=['order_date'])
    .groupby('order_month', as_index=False)
    .agg(order_count=('order_id', 'size'))
    .sort_values('order_month')
)
monthly_order_counts


   order_month  order_count
0      2025-07           19
1      2025-08           26
2      2025-09           12
3      2025-10           22
4      2025-11           23
5      2025-12           23
6      2026-01           26
7      2026-02           19
8      2026-03           22
9      2026-04           26
10     2026-05           20
11     2026-06           24
12     2026-07           12
13     2026-08           21
14     2027-01            1
15     2028-12            1

In [35]:
# 과제 4. 주문 상세 금액(line_total) 상위 10건을 확인합니다.
top_10_order_items = (
    order_items_clean.nlargest(10, 'line_total')
    [['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']]
)
top_10_order_items


     order_item_id  order_id  product_id  quantity  unit_price  line_total
750            751       295          31     100.0    155000.0  15500000.0
737            738       290          96      50.0     96000.0   4800000.0
319            320       129          23       5.0    197000.0    985000.0
9               10         5          65       5.0    193000.0    965000.0
20              21         9          36       5.0    193000.0    965000.0
24              25        11          65       5.0    193000.0    965000.0
116            117        51          65       5.0    193000.0    965000.0
293            294       119          36       5.0    193000.0    965000.0
290            291       117          47       5.0    188000.0    940000.0
10              11         6          97       5.0    187000.0    935000.0

## 저장한 전처리 결과 다시 확인

CSV를 저장한 뒤 다시 읽어 행·열 구조와 날짜 재변환 실패를 확인합니다. 저장 성공만으로 내용이 올바르다고 판단하지 않습니다.


In [36]:
customers_check = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products_check = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders_check = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items_check = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

for name, df in {
    'customers': customers_check,
    'products': products_check,
    'orders': orders_check,
    'order_items': order_items_check,
}.items():
    print(name, 'shape:', df.shape, '| columns:', df.columns.tolist())

orders_check['order_date'] = pd.to_datetime(orders_check['order_date'], errors='coerce')
print('저장 후 order_date 재변환 실패:', int(orders_check['order_date'].isna().sum()))
print('저장 후 order_date 범위:', orders_check['order_date'].min(), '–', orders_check['order_date'].max())


customers shape: (150, 6) | columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products shape: (96, 4) | columns: ['product_id', 'product_name', 'category', 'price']
orders shape: (300, 7) | columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items shape: (752, 6) | columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
저장 후 order_date 재변환 실패: 3
저장 후 order_date 범위: 2025-07-01 00:00:00 – 2028-12-31 00:00:00


## LLM 검토용 프롬프트

실제 이름이나 고객 행은 보내지 않고, 품질 요약과 판단 질문만 제공합니다. 이 프롬프트는 답을 대신 확정시키기보다 처리 기준과 검증 방법을 검토하도록 요청합니다.


In [37]:
llm_review_prompt = f"""
다음 고객 데이터의 결측·변환 요약을 보고 age와 city 처리 기준을 검토해 주세요.
- 원본 행 수: {customers.shape[0]}
- 완전 중복 행: {customers.duplicated().sum()}
- age 숫자 변환 실패/결측: {pd.to_numeric(customers['age'], errors='coerce').isna().sum()}건
- city 결측: {customers['city'].isna().sum()}건
- 적용한 기준: age 결측은 중앙값으로 대체, city 결측은 'Unknown'으로 보존

중앙값 대체, 결측 유지, 행 제외의 장단점을 비교하고,
대표값 대체가 분포와 집단 비교에 미칠 수 있는 영향을 설명해 주세요.
확인해야 할 전후 검증 항목과 업무 담당자에게 물어볼 질문을 제안해 주세요.
실제 고객의 이름이나 식별 정보는 사용하지 말고, 근거가 없는 값을 단정하지 마세요.
"""
print(llm_review_prompt)



다음 고객 데이터의 결측·변환 요약을 보고 age와 city 처리 기준을 검토해 주세요.
- 원본 행 수: 152
- 완전 중복 행: 2
- age 숫자 변환 실패/결측: 6건
- city 결측: 2건
- 적용한 기준: age 결측은 중앙값으로 대체, city 결측은 'Unknown'으로 보존

중앙값 대체, 결측 유지, 행 제외의 장단점을 비교하고,
대표값 대체가 분포와 집단 비교에 미칠 수 있는 영향을 설명해 주세요.
확인해야 할 전후 검증 항목과 업무 담당자에게 물어볼 질문을 제안해 주세요.
실제 고객의 이름이나 식별 정보는 사용하지 말고, 근거가 없는 값을 단정하지 마세요.



## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


# Chapter 05 제출 답안 템플릿

## 0. 제출 정보
- 이름: jasminmuze (GitHub ID)
- GitHub ID: jasminmuze
- 작성일: 2026-10-10
- 최종 제출 URL: https://github.com/jasminmuze/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- Shape: customers 152×6, products 102×4, orders 302×5, order_items 765×5.
- 결측: customers.age 4건, customers.city 2건, 나머지 컬럼은 0건(총 6셀).
- 전체 행 중복: 각 데이터셋 2건씩, 총 8건. 주요 ID 중복도 각 2건씩 확인되며 이 샘플에서는 완전 중복 행과 겹칩니다. `order_items.order_id`처럼 관계를 위한 ID가 반복되는 것은 여러 상품 행이 한 주문에 속할 수 있어 정상일 수 있습니다.
- 타입 후보: age, price, quantity, unit_price와 날짜 컬럼이 문자열로 읽혔습니다. `age`에는 숫자가 아닌 표기와 빈 값이 있고, 가격에는 천 단위 쉼표가 들어간 값이 있습니다.

### 결과 관찰과 판단
원본을 덮어쓰지 않도록 복사본을 만들고, 먼저 키·타입·날짜·참조 관계를 확인했습니다. 잘못된 키는 집계 누락이나 잘못된 조인을 만들 수 있고, 숫자와 날짜가 문자열이면 계산과 기간 분석이 불가능하기 때문입니다. 그 다음 변환 실패, 결측, 이상값 후보를 분리해 각각의 처리 기준을 적용했습니다.

### 업무·분석적 의미와 한계
나이 결측과 도시 결측은 고객 세그먼트 분석에 영향을 줍니다. 주문 상세의 유효하지 않은 상품 참조는 상품별 매출 집계에서 누락 또는 조인 실패를 만들 수 있습니다. 샘플 데이터만으로 결측 발생 이유나 ID 오류의 업무 원인을 확정할 수 없으므로 실제 시스템 로그·담당자 확인이 필요합니다.

## 2. 문자열·타입·결측 처리
- 문자열 컬럼 앞뒤 공백을 제거하고 기존 결측값은 그대로 보존했습니다.
- 숫자 변환 실패: age 6건(빈 값 4 + 숫자가 아닌 값 2), price 2건, quantity 5건, unit_price 2건. 쉼표를 제거한 뒤 숫자로 바꾸고 실패는 `NaN`으로 드러냈습니다.
- 날짜 변환 실패: signup_date 2건, order_date 3건. 날짜는 추정값으로 대체하지 않고 `NaT`로 보존했습니다. 주문 날짜의 변환 가능 범위는 2025-07-01부터 2028-12-31입니다.
- 결측 처리: age는 중앙값 41로 대체, city는 `Unknown`으로 구분했습니다. signup_date와 order_date의 실패 값은 그대로 남겨 후속 분석에서 제외하거나 별도 확인할 수 있게 했습니다.

### 결과 관찰과 판단
중앙값은 일부 극단값의 영향을 평균보다 덜 받지만, 개인의 실제 나이를 복원하는 값은 아닙니다. `Unknown`은 도시를 임의로 추측하지 않으면서 결측 집단을 보존합니다. 대체로 결측 표시가 사라지면 원래 결측 규모가 가려질 수 있어, 대체 전 결측 수를 따로 기록했습니다.

### 업무·분석적 의미와 한계
도시별 고객 수를 볼 때 `Unknown`을 별도 범주로 포함해야 합니다. 나이 중앙값 대체는 연령대 비율·분산을 왜곡할 수 있습니다. 나이 비정상 표기와 날짜 실패 원인은 원천 입력 형식 확인 전까지 확정할 수 없습니다.

## 3. 중복·범주 표준화·이상값 후보
- 완전 중복: 각 원본에서 2행씩 제거했습니다. 처리 후 전체 행 및 주요 ID 중복은 0건입니다.
- 주문 상태 표기를 completed 180건, cancelled 63건, refunded 57건으로 통일했으며 표준 범주 밖 값은 0건입니다.
- 이상값 후보: age 150인 고객 1건, price 변환 실패 2건 및 0 이하 2건, quantity 변환 실패 5건 및 0 이하 3건, unit_price 변환 실패 2건 및 0 이하 1건. `quantity` 50·100인 큰 주문 수량도 상위 금액에서 확인했습니다.

### 결과 관찰과 판단
이상값 후보를 자동으로 오류라고 단정하지 않았습니다. 0원 상품은 증정·행사일 수 있고, 음수 수량은 반품일 수 있으며, 대량 구매는 실제 거래일 수 있습니다. 실습 규칙에 따라 정상 주문 분석용 전처리에서는 숫자 변환 실패와 0 이하 가격·수량·단가 행을 제외했지만, 실무에서는 원본 근거와 업무 정의를 먼저 확인해야 합니다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 customer_id: 2건 (orders에서 customers에 없는 값).
- 없는 order_id: 2건 (order_items에서 orders에 없는 값).
- 없는 product_id: 26건. 이 중 유효하지 않은 가격으로 제외된 상품을 가리키는 행이 포함되어 있고, 9991·9992처럼 원본 상품표에도 없는 ID가 있습니다.
- 파생 컬럼: order_month, order_dayofweek, line_total.
- `line_total = quantity × unit_price`로 계산하고 상위 10건을 확인했습니다. 최고 후보는 수량 100, 단가 155,000인 상세 행(15,500,000)이며 실제 대량 주문인지 확인이 필요합니다.

### 결과 관찰과 판단
필터링 후 부모 테이블의 키가 제거되면 자식 테이블에 고아 참조가 남을 수 있어 PK/FK를 다시 확인했습니다. 26건의 상품 참조 불일치를 그대로 표시해 추가 조치가 필요한 행과 원인을 확인할 수 있게 했습니다. `line_total`은 상세 한 행의 수량×단가이며 할인·취소·환불을 반영한 순매출은 아닙니다.

## 5. 전처리 전/후 비교
| 데이터셋 | 행 수 전→후 | 결측 셀 전→후 | 전체 중복 전→후 | 주요 변화 |
| --- | ---: | ---: | ---: | --- |
| customers | 152→150 | 6→2 | 2→0 | 완전 중복 2행 제거, age 결측/비숫자 중앙값 대체, city 2건 Unknown, signup_date 실패 2건 유지 |
| products | 102→96 | 0→0 | 2→0 | 중복 2행 제거, 가격 변환 실패 2행과 0 이하 2행 제외 |
| orders | 302→300 | 0→9 | 2→0 | 중복 2행 제거, 날짜 실패 3건은 order_date/month/dayofweek에 남음, 파생 열 2개 추가 |
| order_items | 765→752 | 0→0 | 2→0 | 중복 2행, quantity 실패/0 이하 8행, unit_price 실패/0 이하 3행 제외; line_total 추가 |

행 수 감소는 중복 제거 및 명시한 숫자/가격 기준에 따른 결과입니다. 주문 날짜 결측 3건은 행을 지우지 않아 월 집계에서 제외됩니다. 정상 주문을 대상으로 한 분석 규칙이라도 제외한 행은 원본과 연결해 기록해야 합니다.

## 6. 재현 가능한 전처리 확인
- 생성 파일: `data/processed/` 아래 clean CSV 4개.
- 요약 보고서: `reports/ch05_preprocessing_summary.md`.
- `python scripts/preprocess_data.py` 실행 결과: 오류 없이 완료, 같은 행 수와 관계 검증 건수(2, 2, 26)를 출력했습니다.
- 저장 CSV 재읽기: 네 파일 모두 기대 행·열 구조를 확인했고, 저장 후 order_date 재변환 실패는 3건으로 원본 변환 결과와 일치했습니다.
- 재실행 시 원본은 변경하지 않고 같은 규칙으로 결과를 다시 생성합니다. 실행 기록과 요약을 남겨 환경이 달라져도 변경 내용을 비교할 수 있습니다.

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본을 보존하고 복사본에서 처리한다.
2. 변환 실패·중복·이상값은 처리 전후 건수를 기록한다.
3. 행을 제외한 뒤에는 PK/FK 관계와 저장된 결과를 다시 검증한다.

### 가장 위험한 자동 처리와 이유
나이 결측을 중앙값으로 대체하는 일입니다. 결측이 특정 연령 집단에서 더 자주 발생했다면 중앙값 대체가 실제 분포와 집단 간 차이를 감추고 분산을 줄일 수 있습니다. 원인 분석 및 결측 표시 보존 여부를 먼저 검토해야 합니다.

### 다음 EDA에서 주의할 특성
나이 150 후보, 수량 50·100 상세행, 0 이하 가격·수량·단가 제외의 영향, 2027·2028년 주문일, 날짜 변환 실패 3건, 26건 상품 참조 불일치가 월별·상품별 집계에 미치는 영향을 확인합니다. `line_total` 합계를 매출이나 순매출로 바로 해석하지 않습니다.

### 현재 전처리의 한계
이 기준은 실습용입니다. 비정상 거래나 고아 키의 업무 원인은 확인하지 않았고, 가격·수량 조건으로 제외한 행도 별도 격리 파일로 보존하지 않았습니다. 실제 분석 전에 환불·취소·행사·대량구매의 정의와 주문 날짜의 유효 기간을 담당자와 확인해야 합니다.

## Chapter 05 완료 Evidence
- Notebook 코드 실행: PASS
- raw 원본 보존: PASS
- processed CSV 4개 생성: PASS
- 숫자/날짜 변환 실패 점검: PASS
- 전체 중복/키 중복 점검: PASS
- PK/FK 관계 재검증: PASS (불일치 2, 2, 26건을 확인·기록)
- 파생 컬럼 생성: PASS
- `python scripts/preprocess_data.py` 재실행: PASS
- 요약 보고서 생성: PASS
- 저장 결과 다시 읽어 구조와 날짜를 확인: PASS
- LLM 검토 프롬프트 작성: PASS

## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 작성했습니다.
- [x] clean CSV와 스크립트 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret을 노트북에 포함하지 않았습니다.
- [x] Notebook을 실행해 결과를 저장했습니다.
- [x] GitHub 노트북 URL을 LMS에 제출합니다.
